<style>
.jp-Notebook, .notebook-container, .markdown-body {font-family: Arial, Helvetica, sans-serif;}
.jp-MarkdownOutput, .text_cell_render {font-size: 18px; line-height: 1.65;}
h1 {font-size: 2.25rem !important; margin-top: 0.35em !important;}
h2 {font-size: 1.65rem !important; margin-top: 1.35em !important;}
h3 {font-size: 1.25rem !important; margin-top: 1.1em !important;}
table {font-size: 0.95em;}
blockquote {border-left: 4px solid #aaa; padding-left: 1rem;}
</style>


# 05 · Simulación de Monte Carlo

<p><a href="https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/notebooks/05_Simulacion_Monte_Carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></p>

> **Objetivo:** usar muestreo aleatorio repetido para proyectar muchos escenarios, convertir incertidumbre en una distribución de resultados y evaluar la estabilidad de la simulación.

Esta notebook está diseñada para **jugar deliberadamente con la aleatoriedad**: una seed fija, una seed cambiante y múltiples seeds.


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/main/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/Ciclo_MC.png" alt="Ciclo Monte Carlo" width="650">


## 1. La idea

Un presupuesto tradicional entrega un número. Monte Carlo pregunta:

> **¿Qué futuros podrían ocurrir, con qué frecuencia y qué decisión tomaríamos ante ese rango?**

**modelo → entradas inciertas → muestreo → repetir → distribución de salidas → decisión**

### 💡 IDEA
Una simulación no predice exactamente el futuro. Construye un laboratorio de escenarios coherentes con los supuestos que definimos.


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/main/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/MC_Aplicaciones.png" alt="Aplicaciones Monte Carlo" width="680">


## 2. Calentamiento: aproximar π

### ¿Por qué este ejercicio?
π es un ejemplo visual y sencillo para entender la mecánica básica:

1. generar valores aleatorios;
2. evaluar una condición;
3. repetir muchas veces;
4. usar una proporción observada para aproximar una cantidad.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng=np.random.default_rng(42)

n=20_000
x=rng.uniform(-1,1,n)
y=rng.uniform(-1,1,n)

dentro=x**2+y**2<=1
pi_estimado=4*dentro.mean()

print(f"π estimado: {pi_estimado:.5f}")
print(f"error absoluto: {abs(np.pi-pi_estimado):.5f}")


## 3. Ejemplo visual: un apostador y muchos futuros posibles

### ¿Por qué conservar este ejemplo?

Este ejercicio hace visible una idea central de Monte Carlo: **muchas trayectorias pueden partir exactamente del mismo punto y terminar en lugares muy distintos**.

Usaremos una apuesta fija con probabilidad de ganar de 49% y pago simétrico 1:1. No es una recomendación de apuestas ni una estrategia para ganar; es un **paseo aleatorio didáctico** para observar cómo se acumula la incertidumbre ronda tras ronda.

La pregunta es:

> **Si 1,000 apostadores empiezan con el mismo capital y enfrentan las mismas reglas probabilísticas, ¿qué nube de futuros aparece?**

Con `p_ganar = 0.49`, el juego tiene una ligera desventaja matemática para el apostador. Aun así, algunas trayectorias terminarán arriba del capital inicial por puro azar. Eso permite distinguir entre **una corrida favorable** y **el comportamiento del sistema**.


In [ ]:
def simular_apostador(
    seed=42,
    n_sim=1_000,
    capital_inicial=1_000,
    apuesta=50,
    n_apuestas=100,
    p_ganar=0.49
):
    """Simula muchas trayectorias de una apuesta fija.

    Si el capital cae por debajo de la apuesta mínima, esa trayectoria deja de apostar.
    """
    rng = np.random.default_rng(seed)

    trayectorias = np.empty((n_sim, n_apuestas + 1), dtype=float)
    trayectorias[:, 0] = capital_inicial

    for ronda in range(1, n_apuestas + 1):
        capital_previo = trayectorias[:, ronda - 1]
        pueden_apostar = capital_previo >= apuesta

        gana = rng.random(n_sim) < p_ganar
        cambio = np.where(gana, apuesta, -apuesta)

        trayectorias[:, ronda] = np.where(
            pueden_apostar,
            capital_previo + cambio,
            capital_previo
        )

    return trayectorias


trayectorias_apuesta = simular_apostador(seed=42)
finales_apuesta = trayectorias_apuesta[:, -1]

print(f"Capital final medio: ${finales_apuesta.mean():,.0f}")
print(f"Mediana final: ${np.median(finales_apuesta):,.0f}")
print(f"Terminan arriba del capital inicial: {(finales_apuesta > 1_000).mean():.1%}")
print(f"Terminan abajo del capital inicial: {(finales_apuesta < 1_000).mean():.1%}")
print(f"Quiebran: {(finales_apuesta == 0).mean():.1%}")


In [ ]:
rondas = np.arange(trayectorias_apuesta.shape[1])

p10_apuesta = np.percentile(trayectorias_apuesta, 10, axis=0)
p50_apuesta = np.percentile(trayectorias_apuesta, 50, axis=0)
p90_apuesta = np.percentile(trayectorias_apuesta, 90, axis=0)

plt.figure(figsize=(10, 6))

# Dibujamos sólo una muestra de trayectorias para no saturar la figura.
plt.plot(rondas, trayectorias_apuesta[:180].T, alpha=.08)

# Resumimos la nube completa con percentiles.
plt.fill_between(rondas, p10_apuesta, p90_apuesta, alpha=.25, label="P10–P90")
plt.plot(rondas, p50_apuesta, linewidth=2, label="P50")
plt.axhline(1_000, linestyle="--", label="Capital inicial")

plt.xlabel("Número de apuestas")
plt.ylabel("Capital")
plt.title("Un mismo punto de partida, muchos futuros posibles")
plt.legend()
plt.show()


### Qué mirar en la gráfica

Todas las trayectorias parten de **$1,000**, pero la incertidumbre se abre conforme avanzan las rondas.

- Una línea individual puede parecer extraordinariamente buena o mala.
- La banda P10–P90 muestra la región donde termina la mayor parte de los escenarios.
- P50 resume la trayectoria mediana.
- El hecho de que existan ganadores no elimina la desventaja estructural de una probabilidad de 49%.

### 🔬 PRUEBA

Vuelve a ejecutar cambiando sólo una cosa:

- `seed=42` → misma nube cada vez;
- `seed=None` → una nube nueva;
- `p_ganar=0.50` → juego aproximadamente neutral;
- `p_ganar=0.51` → ligera ventaja positiva.

Observa cómo cambia la **nube completa**, no sólo una trayectoria.


## 4. Caso principal: proyectar muchos escenarios de costo

Modelaremos un proyecto durante 12 meses con:

- costo fijo mensual;
- infraestructura variable;
- mantenimiento con distribución asimétrica;
- eventos extraordinarios con cierta probabilidad.

La pregunta no es "¿cuánto costará exactamente?", sino:

> **¿cómo se distribuyen miles de trayectorias posibles y qué riesgo hay de exceder un presupuesto?**

Esto es el corazón del ejercicio: **proyectar muchos futuros**, no sólo calcular una media.


In [ ]:
def simular_proyecto(seed, n_sim=20_000, meses=12):
    # seed puede ser un entero reproducible o None para una secuencia nueva.
    rng=np.random.default_rng(seed)

    fijo_mensual=45_000

    infraestructura=rng.normal(
        loc=8_000,
        scale=1_500,
        size=(n_sim, meses)
    )

    mantenimiento=rng.lognormal(
        mean=np.log(3_000),
        sigma=.35,
        size=(n_sim, meses)
    )

    ocurre_incidente=rng.random((n_sim, meses)) < .08
    costo_incidente=ocurre_incidente * rng.uniform(
        8_000,
        25_000,
        size=(n_sim, meses)
    )

    costo_mensual=(
        fijo_mensual
        + infraestructura
        + mantenimiento
        + costo_incidente
    )

    costo_acumulado=np.cumsum(costo_mensual, axis=1)

    return costo_mensual, costo_acumulado


### 4.1 Una seed fija: mismo experimento, mismo resultado

### ¿Por qué?
Cuando desarrollamos o comparamos código necesitamos separar cambios del modelo de cambios causados por el azar. Una seed fija nos permite **reproducir exactamente la misma simulación**.


In [ ]:
presupuesto=700_000

mensual_fijo, acumulado_fijo = simular_proyecto(seed=42, n_sim=20_000)
costos_fijos = acumulado_fijo[:, -1]

print(f"Media anual: ${costos_fijos.mean():,.0f}")
print(f"Mediana anual: ${np.median(costos_fijos):,.0f}")
print(f"P95: ${np.percentile(costos_fijos,95):,.0f}")
print(f"P(exceder presupuesto): {(costos_fijos>presupuesto).mean():.1%}")


### 4.2 Visualizar cientos de futuros: trayectorias + banda de escenarios

Aquí no queremos una sola curva. Queremos ver **muchos caminos posibles** del costo acumulado.

La banda P10–P90 contiene el 80% central de los escenarios simulados; P50 es la trayectoria mediana.


In [ ]:
meses=np.arange(1,13)

# Mostramos sólo una parte de las trayectorias para no saturar la figura.
plt.figure(figsize=(10,6))
plt.plot(meses, acumulado_fijo[:120].T, alpha=.07)

p10=np.percentile(acumulado_fijo,10,axis=0)
p50=np.percentile(acumulado_fijo,50,axis=0)
p90=np.percentile(acumulado_fijo,90,axis=0)

plt.fill_between(meses,p10,p90,alpha=.25,label="P10–P90")
plt.plot(meses,p50,linewidth=2,label="P50")
plt.axhline(presupuesto,linestyle="--",label="Presupuesto anual")

plt.xlabel("Mes")
plt.ylabel("Costo acumulado")
plt.title("Proyección de múltiples escenarios")
plt.legend()
plt.show()


### 4.3 Distribución del resultado final

Las trayectorias muestran **cómo** se llega a distintos futuros. El histograma resume **dónde terminan** los escenarios.


In [ ]:
plt.hist(costos_fijos,bins=45)
plt.axvline(presupuesto,linestyle="--",label="Presupuesto")
plt.xlabel("Costo anual")
plt.ylabel("Escenarios")
plt.title("Distribución de costos al final de 12 meses")
plt.legend()
plt.show()


## 5. Jugar con la seed

### Caso A · seed fija
`seed=42` reproduce el mismo stream pseudoaleatorio.

### Caso B · seed cambiante
`seed=None` obtiene entropía fresca del sistema. Si vuelves a ejecutar la celda, los escenarios cambian.

### Caso C · múltiples seeds
Repetimos el experimento con **streams distintos** para medir sensibilidad a la aleatoriedad.

### ⚠️ Matiz importante
Cambiar de seed no corrige un modelo equivocado. Sólo cambia la realización aleatoria **dentro del mismo modelo**.


In [ ]:
# Seed cambiante: ejecuta esta celda varias veces y observa cómo varía el resultado.
_, acumulado_nuevo = simular_proyecto(seed=None, n_sim=5_000)
costos_nuevos = acumulado_nuevo[:, -1]

print(f"Media: ${costos_nuevos.mean():,.0f}")
print(f"P95: ${np.percentile(costos_nuevos,95):,.0f}")
print(f"Riesgo de exceso: {(costos_nuevos>presupuesto).mean():.1%}")


## 6. Múltiples seeds: estabilidad del resultado

### ¿Por qué?
Una sola corrida puede ser ligeramente optimista o pesimista por variabilidad Monte Carlo.

Para crear streams reproducibles y muy probablemente no solapados, usamos `SeedSequence.spawn()`.


In [ ]:
# Seed raíz reproducible para crear varios streams independientes.
raiz = np.random.SeedSequence(2026)
secuencias = raiz.spawn(20)

resultados=[]

for i, secuencia in enumerate(secuencias):
    _, acumulado = simular_proyecto(seed=secuencia, n_sim=10_000)
    costos = acumulado[:, -1]

    resultados.append({
        "corrida": i+1,
        "media": costos.mean(),
        "p95": np.percentile(costos,95),
        "riesgo_exceso": (costos>presupuesto).mean()
    })

estabilidad=pd.DataFrame(resultados)
estabilidad.head()


In [ ]:
estabilidad[["media","p95","riesgo_exceso"]].agg(["mean","std","min","max"])


### Interpretación
Si la desviación entre corridas es pequeña, nuestra conclusión es relativamente estable frente al stream aleatorio.

Si cambia demasiado, tenemos dos opciones principales:

1. aumentar `n_sim`;
2. revisar si el evento que estimamos es muy raro o si el modelo necesita otra estrategia.

**Múltiples seeds miden robustez; más simulaciones reducen el error Monte Carlo.**


## 7. Convergencia: ¿cuántas simulaciones son suficientes?

No existe un número universal. Depende de la precisión requerida y del evento que estimas.

Aquí observamos cómo cambia el riesgo estimado al aumentar el número de escenarios.


In [ ]:
tamanos=[500,1_000,5_000,10_000,50_000]
riesgos=[]

for n_sim in tamanos:
    _, acumulado=simular_proyecto(seed=123,n_sim=n_sim)
    costos=acumulado[:, -1]
    riesgos.append((costos>presupuesto).mean())

convergencia=pd.DataFrame({
    "n_sim":tamanos,
    "riesgo_exceso":riesgos
})
convergencia


In [ ]:
plt.plot(convergencia["n_sim"],convergencia["riesgo_exceso"],marker="o")
plt.xscale("log")
plt.xlabel("Número de escenarios")
plt.ylabel("P(exceder presupuesto)")
plt.title("Convergencia del estimador")
plt.show()


## 8. Múltiples seeds y Machine Learning

La misma idea aparece en ML porque puede haber aleatoriedad en:

- `train/test split`;
- inicialización de pesos;
- orden de minibatches;
- bagging / random forests;
- búsqueda de hiperparámetros;
- subsampling.

Evaluar una sola seed puede hacer que un modelo parezca mejor o peor por una partición o inicialización afortunada.

Cuando sea apropiado, repetir con varias seeds y reportar **media ± desviación estándar** ayuda a medir estabilidad.

### ⚠️ ERROR ÚTIL
Múltiples seeds **no corrigen**:

- muestra sesgada;
- data leakage;
- errores de medición;
- variables omitidas;
- un modelo mal especificado.

Eso es un problema de **datos, diseño o modelo**, no del generador aleatorio.


## 9. Mini reto: laboratorio de escenarios

Prueba estas tres versiones:

1. **fija:** `seed=42`;
2. **cambiante:** `seed=None`, ejecutándola varias veces;
3. **múltiple:** 20 streams con `SeedSequence.spawn()`.

Después modifica la probabilidad de incidente de 8% a 3%, 12% y 20%.

Para cada caso compara:

- P50;
- P90/P95;
- probabilidad de exceder presupuesto;
- variación entre seeds.

Finalmente explica qué decisión presupuestal tomarías y **qué supuesto del modelo revisarías primero**.


## 📚 Material adicional
- [NumPy · Generator](https://numpy.org/doc/stable/reference/random/generator.html) — generador recomendado y reproducibilidad.
- [NumPy · SeedSequence](https://numpy.org/doc/stable/reference/random/bit_generators/generated/numpy.random.SeedSequence.html) — creación de streams reproducibles e independientes.
- [SciPy · Statistics, resampling and Monte Carlo](https://docs.scipy.org/doc/scipy/tutorial/stats.html) — Monte Carlo y métodos de remuestreo.
- [NIST/SEMATECH Engineering Statistics Handbook](https://www.nist.gov/programs-projects/nistsematech-engineering-statistics-handbook) — contexto estadístico general para incertidumbre y diseño.

### 🧾 Términos clave
**escenario · simulación · seed · stream aleatorio · convergencia · error Monte Carlo · percentil · riesgo · sensibilidad**


## Qué sigue
Terminamos con un puente práctico: obtener datos desde web o APIs antes de analizarlos.
